In [0]:
USE CATALOG smart_odoo;
 
-- =====================================================
-- VIEW 1: gold.vw_crm_kpi_summary
-- Single-row — feeds all 4 KPI cards
-- Total Customers  → fact_sales distinct customer_id (real customers)
-- Active Deals     → fact_crm_pipeline open opportunities
-- Total Deal Value → won deals expected_revenue
-- Conversion Rate  → won / total opportunities
-- =====================================================
 
CREATE OR REPLACE VIEW gold.vw_crm_kpi_summary AS
WITH customers AS
(
    SELECT DISTINCT customer_id
    FROM gold.fact_sales
),
customers_this_month AS
(
    SELECT DISTINCT customer_id
    FROM gold.fact_sales
    WHERE CAST(order_date AS DATE) >= DATE_TRUNC('month', current_date())
),
active_deals AS
(
    SELECT COUNT(DISTINCT lead_id) AS cnt
    FROM gold.fact_crm_pipeline
    WHERE active  = TRUE
      AND is_won  = FALSE
      AND is_lost = FALSE
),
current_month_value AS
(
    SELECT COALESCE(SUM(expected_revenue), 0) AS value
    FROM gold.fact_crm_pipeline
    WHERE is_won = TRUE
      AND CAST(date_closed AS DATE) >= DATE_TRUNC('month', current_date())
),
last_month_value AS
(
    SELECT COALESCE(SUM(expected_revenue), 0) AS value
    FROM gold.fact_crm_pipeline
    WHERE is_won = TRUE
      AND CAST(date_closed AS DATE) >= DATE_TRUNC('month', ADD_MONTHS(current_date(), -1))
      AND CAST(date_closed AS DATE) <  DATE_TRUNC('month', current_date())
),
current_month_conv AS
(
    SELECT
        COUNT(DISTINCT CASE WHEN is_won = TRUE THEN lead_id END)    AS won,
        COUNT(DISTINCT lead_id)                                      AS total
    FROM gold.fact_crm_pipeline
    WHERE CAST(date_open AS DATE) >= DATE_TRUNC('month', current_date())
),
last_month_conv AS
(
    SELECT
        COUNT(DISTINCT CASE WHEN is_won = TRUE THEN lead_id END)    AS won,
        COUNT(DISTINCT lead_id)                                      AS total
    FROM gold.fact_crm_pipeline
    WHERE CAST(date_open AS DATE) >= DATE_TRUNC('month', ADD_MONTHS(current_date(), -1))
      AND CAST(date_open AS DATE) <  DATE_TRUNC('month', current_date())
)
SELECT
    -- KPI 1: Total Customers (real customers from fact_sales)
    (SELECT COUNT(*) FROM customers)                                AS total_customers,
    (SELECT COUNT(*) FROM customers_this_month)                    AS customers_added_this_month,
 
    -- KPI 2: Active Deals
    (SELECT cnt FROM active_deals)                                 AS active_deals,
 
    -- KPI 3: Total Deal Value
    ROUND((SELECT value FROM current_month_value), 2)              AS total_deal_value,
    ROUND((SELECT value FROM last_month_value), 2)                 AS last_month_deal_value,
    ifnull(ROUND(
        CASE
            WHEN (SELECT value FROM last_month_value) > 0
            THEN (
                    (SELECT value FROM current_month_value)
                  - (SELECT value FROM last_month_value)
                 ) / (SELECT value FROM last_month_value) * 100
            ELSE NULL
        END, 2
    ),0)                                                              AS deal_value_pct_change,
 
    -- KPI 4: Conversion Rate
    ROUND(
        CASE
            WHEN (SELECT total FROM current_month_conv) > 0
            THEN (SELECT won  FROM current_month_conv)
               / (SELECT total FROM current_month_conv) * 100
            ELSE 0
        END, 2
    )                                                              AS conversion_rate_pct,
 
    ROUND(
        CASE
            WHEN (SELECT total FROM last_month_conv) > 0
            THEN (SELECT won  FROM last_month_conv)
               / (SELECT total FROM last_month_conv) * 100
            ELSE 0
        END, 2
    )                                                              AS last_month_conversion_rate_pct,
 
    ifnull(ROUND(
        CASE
            WHEN (SELECT total FROM last_month_conv) > 0
            THEN
            (
                (SELECT won FROM current_month_conv)
              / NULLIF((SELECT total FROM current_month_conv), 0)
              - (SELECT won FROM last_month_conv)
              / NULLIF((SELECT total FROM last_month_conv), 0)
            ) * 100
            ELSE NULL
        END, 2
    ),0)                                                         AS conversion_rate_change;
 
 


In [0]:
USE CATALOG smart_odoo;
 
CREATE OR REPLACE VIEW gold.vw_crm_customer_growth AS
WITH first_order AS
(
    -- each customer's first ever order date
    SELECT
        customer_id,
        MIN(CAST(order_date AS DATE)) AS first_order_date
    FROM gold.fact_sales
    GROUP BY customer_id
)
SELECT
    dd.month_short                                             AS month_name,
    dd.month_number,
    dd.year_number,
 
    COUNT(DISTINCT fo.customer_id)                                  AS new_customers,
 
    SUM(COUNT(DISTINCT fo.customer_id))
        OVER (ORDER BY dd.month_number
              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)     AS cumulative_customers
 
FROM first_order fo
INNER JOIN gold.dim_date dd
    ON dd.full_date = fo.first_order_date
 
WHERE dd.year_number = YEAR(current_date())
 
GROUP BY
    dd.month_short,
    dd.month_number,
    dd.year_number
 
ORDER BY dd.month_number ASC;


In [0]:
USE CATALOG smart_odoo;
 

CREATE OR REPLACE VIEW gold.vw_crm_customer_segments AS
WITH customer_sales AS
(
    SELECT
        fs.customer_id,
        SUM(fs.net_revenue)         AS total_revenue,
        COUNT(DISTINCT fs.order_id) AS total_orders
    FROM gold.fact_sales fs
    GROUP BY fs.customer_id
)
SELECT
    CASE
        WHEN cs.total_revenue >= 100000              THEN 'Enterprise'
        WHEN cs.total_revenue >= 10000               THEN 'Mid-Market'
        WHEN cs.total_revenue >= 1000                THEN 'Small Business'
        ELSE                                              'Starter'
    END                                                             AS segment,
 
    COUNT(DISTINCT dp.partner_id)                                   AS total_customers,
    ROUND(SUM(cs.total_revenue), 2)                                 AS total_revenue,
    ROUND(AVG(cs.total_orders), 1)                                  AS avg_orders_per_customer
 
FROM customer_sales cs
INNER JOIN gold.dim_partner dp
    ON dp.partner_id = cs.customer_id
 
WHERE dp.active = TRUE
 
GROUP BY
    CASE
        WHEN cs.total_revenue >= 100000              THEN 'Enterprise'
        WHEN cs.total_revenue >= 10000               THEN 'Mid-Market'
        WHEN cs.total_revenue >= 1000                THEN 'Small Business'
        ELSE                                              'Starter'
    END
 
ORDER BY total_revenue DESC;
 


In [0]:
USE CATALOG smart_odoo;
 
-- =====================================================
-- VIEW 4: gold.vw_crm_sales_pipeline
-- Bar Chart — X = stage name, Y = deal count
-- Ordered by stage sequence for correct funnel order
-- =====================================================
 
CREATE OR REPLACE VIEW gold.vw_crm_sales_pipeline AS
SELECT
    f.stage_id,
    f.stage_name,
 
    COUNT(DISTINCT f.lead_id)                                       AS deal_count,
    ROUND(SUM(f.expected_revenue), 2)                               AS total_expected_revenue,
    ROUND(AVG(f.probability), 2)                                    AS avg_probability,
    ROUND(AVG(f.day_open), 2)                                       AS avg_days_in_stage
 
FROM gold.fact_crm_pipeline f
 
WHERE f.active  = TRUE
  AND f.is_lost = FALSE
 
GROUP BY
    f.stage_id,
    f.stage_name
 
-- Order by avg probability ascending = natural funnel order
-- Lead(~10%) → Qualified(~25%) → Proposal(~50%) → Negotiation(~75%) → Won(100%)
ORDER BY avg_probability ASC;
 


In [0]:
USE CATALOG smart_odoo;
 
-- =====================================================
-- VIEW 5: gold.vw_crm_customer_list
-- Table view — one row per real customer
-- Source of truth: fact_sales (real customers only)
-- Enriched with: dim_partner (contact info)
--                fact_crm_pipeline (deals + pipeline value)
-- =====================================================
 
CREATE OR REPLACE VIEW gold.vw_crm_customer_list AS
WITH customer_sales AS
(
    -- aggregate sales per customer
    SELECT
        customer_id,
        COUNT(DISTINCT order_id)    AS total_orders,
        ROUND(SUM(net_revenue), 2)  AS total_revenue,
        MAX(order_date)             AS last_order_date,
        MIN(order_date)             AS first_order_date
    FROM gold.fact_sales
    GROUP BY customer_id
),
customer_pipeline AS
(
    -- aggregate CRM deals per partner
    SELECT
        partner_id,
        COUNT(DISTINCT lead_id)             AS total_deals,
        ROUND(SUM(expected_revenue), 2)     AS pipeline_value,
        COUNT(DISTINCT CASE
            WHEN active = TRUE
             AND is_won  = FALSE
             AND is_lost = FALSE
            THEN lead_id END)               AS open_deals
    FROM gold.fact_crm_pipeline
    GROUP BY partner_id
)
SELECT
    -- Customer ID
    dp.partner_id                                                   AS customer_id,
 
    -- Name & Contact from dim_partner
    dp.partner_name                                                 AS customer_name,
    dp.email,
    dp.phone,
 
    -- Status based on purchase activity from fact_sales
    CASE
        WHEN dp.active = FALSE                                      THEN 'Inactive'
        WHEN CAST(cs.last_order_date AS DATE)
             >= ADD_MONTHS(current_date(), -6)                      THEN 'Active'
        ELSE                                                             'Inactive'
    END                                                             AS status,
 
    -- Sales metrics from fact_sales
    COALESCE(cs.total_orders, 0)                                    AS total_orders,
    COALESCE(cs.total_revenue, 0)                                   AS total_value,
    cs.last_order_date,
    cs.first_order_date,
 
    -- Pipeline metrics from fact_crm_pipeline
    COALESCE(cp.total_deals, 0)                                     AS total_deals,
    COALESCE(cp.open_deals, 0)                                      AS open_deals,
    COALESCE(cp.pipeline_value, 0)                                  AS pipeline_value,
 
    -- Extra info (customer_rank kept for reference only — not used as filter)
    dp.customer_rank,
    dp.is_company
 
FROM gold.dim_partner dp
 
-- only real customers
INNER JOIN customer_sales cs
    ON cs.customer_id = dp.partner_id
 
-- optionally enrich with CRM pipeline (LEFT = customers without leads still show)
LEFT JOIN customer_pipeline cp
    ON cp.partner_id = dp.partner_id
 
-- No customer_rank filter — fact_sales INNER JOIN is the source of truth
-- Any partner who placed an order is a customer regardless of rank
 
ORDER BY cs.total_revenue DESC;


In [0]:
Select * from smart_odoo.gold.vw_crm_kpi_summary;
-------------------------------------------------
Select * from smart_odoo.gold.vw_crm_customer_growth;
-------------------------------------------------
Select * from smart_odoo.gold.vw_crm_customer_segments;
-------------------------------------------------
Select * from smart_odoo.gold.vw_crm_sales_pipeline;
-------------------------------------------------
Select * from smart_odoo.gold.vw_crm_customer_list;